In [ ]:
from typing import Annotated, Sequence, TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel
from dotenv import load_dotenv 

load_dotenv()  

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    next_node : str 

In [ ]:
llm = init_chat_model(model_provider="google_genai", 
                        model ="gemini-2.5-flash",
                        temperature=0.7,
                        max_output_tokens=1024,
                        top_p=0.95,
                        top_k=40,
                        timeout = 30,
                        max_retries = 3,
                        load_env=True)

In [ ]:
@tool
def add(a: float , b:float) -> float:
    """Add two numbers"""
    return a + b

In [ ]:
math_agent = create_react_agent(
    llm, 
    tools=[add], 
    prompt="You are a precise calculation assistant. Use your tools for math."
)

In [ ]:
writer_agent = create_react_agent(
    llm,
    tools = [],
    prompt="You are a technical educator. Explain numerical results in plain terms."
)

In [ ]:
class RouterDecision(BaseModel):
    next : str

supervisor_llm = llm.with_structured_output(RouterDecision)

In [ ]:
def supervisor_node(state: AgentState) -> dict:

    prompt = (
        "Given the conversation below, decide who should act next: "
        "'math' (to calculate), 'writer' (to format/explain), or 'FINISH'.\n\n"
        f"Messages: {state['messages']}"
    )

    decision = supervisor_llm.invoke(prompt)
    return {"next_node" : decision.next}

In [ ]:
def math_node(state: AgentState) -> dict:
    result = math_agent.invoke({"messages":state['messages']})
    return {"messages" : [result["message"][-1]]}

In [ ]:
def writer_node(state: AgentState) -> dict:
    result = writer_agent.invoke({"messages": state["messages"]})
    return {"messages": [result["messages"][-1]]}

In [ ]:
builder = StateGraph(AgentState)

In [ ]:
builder.add_node("supervisor", supervisor_node)
builder.add_node("math", math_node)
builder.add_node("writer", writer_node)

In [ ]:
builder.add_edge(START, "supervisor")

In [ ]:
builder.add_conditional_edges(
    "supervisor",
    lambda state: state["next_node"],
    {
        "math": "math",
        "writer": "writer",
        "FINISH": END,
    },
)

In [ ]:
builder.add_edge("math", "supervisor")
builder.add_edge("writer", "supervisor")

multi_agent_graph = builder.compile()

In [ ]:
inputs = {
    "messages": [HumanMessage(content="Calculate 948 + 352, then explain the significance of 1300 in a brief sentence.")]
}

for chunk in multi_agent_graph.stream(inputs, stream_mode="updates"):
    print(chunk)